# Creating New Features

**Topic:** Feature Engineering

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, HTML
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** three patterns for creating new features: ratios, differences, and domain-inspired combinations
- **Evaluate** whether a new feature actually adds predictive signal, instead of assuming it does
- **Identify** when an engineered feature is leaking the value you are trying to predict

> **Tip:** Every feature below is scored against the same number: how strongly it lines up with price. Watch that number in the widget title as you switch features — only one of the four choices actually beats the raw columns it came from.

---
## How we got here

In **[01_what_is_feature_engineering.ipynb](01_what_is_feature_engineering.ipynb)** you saw why raw features often fail a model, and you saw one engineered feature, `dist_midtown`, beat both of the raw columns it was built from.

Now you will build features yourself, learn the patterns behind them, and learn to check whether a new feature is actually adding signal or just rearranging the columns you already had.

---
## Why this matters for data science

Raw features record what happened. Engineered features are supposed to explain more. The only way to know if a new one actually does is to measure it: does it line up with price any more strongly than the raw columns it was built from?

We will use one number for that: the correlation between a feature and price. It runs from 0 to 1. 0 means the feature tells you nothing about price. In this dataset, anything below about 0.05 is indistinguishable from nothing at all.

The best new features come from asking what a domain expert knows about the real world that the raw numbers, on their own, do not capture. But "I engineered a feature" is not the same as "I engineered a useful feature." You still have to check.

---
## Try it yourself

In [2]:
_airbnb_w2 = pd.read_csv('../../data/nyc_airbnb.csv')
_airbnb_w2 = _airbnb_w2[_airbnb_w2['price'] > 0].dropna(
    subset=['number_of_reviews', 'availability_365', 'latitude', 'longitude',
            'calculated_host_listings_count']).copy()
_y_price_w2 = np.log1p(_airbnb_w2['price'])

TIMES_SQUARE_LAT, TIMES_SQUARE_LON = 40.7580, -73.9855
_airbnb_w2['dist_midtown'] = np.sqrt(
    (_airbnb_w2['latitude']  - TIMES_SQUARE_LAT) ** 2 +
    (_airbnb_w2['longitude'] - TIMES_SQUARE_LON) ** 2
)
_airbnb_w2['availability_ratio'] = _airbnb_w2['availability_365'] / 365
_airbnb_w2['calc_diff']          = _airbnb_w2['calculated_host_listings_count'] - 1
_airbnb_w2['price_per_review']   = _airbnb_w2['price'] / (_airbnb_w2['number_of_reviews'] + 1)

CATEGORY_COLOR = {"win": PALETTE["accent"], "none": PALETTE["muted"], "leak": PALETTE["secondary"]}
CATEGORY_LABEL = {"win": "REAL WIN", "none": "NO CHANGE", "leak": "LEAK"}

_FEATURES_W2 = {
    "dist_midtown": {
        "raw_cols": ["latitude", "longitude"],
        "category": "win",
        "caption": (
            "dist_midtown combines two raw columns and beats both of them. Latitude alone "
            "reaches |r| = {r0:.3f}, longitude alone reaches |r| = {r1:.3f}, and this one new "
            "column reaches |r| = {eng:.3f} — a single number for how far a listing sits from "
            "the middle of Manhattan."
        ),
    },
    "availability_ratio": {
        "raw_cols": ["availability_365"],
        "category": "none",
        "caption": (
            "Dividing availability_365 by the constant 365 cannot create new information. "
            "Raw |r| = {r0:.6f}, engineered |r| = {eng:.6f} — the same number, to within "
            "floating-point noise."
        ),
    },
    "calc_diff": {
        "raw_cols": ["calculated_host_listings_count"],
        "category": "none",
        "caption": (
            "Subtracting the constant 1 from calculated_host_listings_count cannot create new "
            "information either. Raw |r| = {r0:.6f}, engineered |r| = {eng:.6f} — same story, "
            "different operation."
        ),
    },
    "price_per_review": {
        "raw_cols": ["number_of_reviews"],
        "category": "leak",
        "caption": (
            "This looks like the best feature on the page, |r| = {eng:.3f} versus {r0:.4f} for "
            "number_of_reviews alone. That is because price is sitting inside its own formula. "
            "It cannot be used to predict price."
        ),
    },
}

out_new_feat = Output()
_initialized = False

feat_dd_w2 = Dropdown(
    options=list(_FEATURES_W2.keys()), value="dist_midtown",
    description="New feature:", style={"description_width": "95px"},
    layout=widgets.Layout(width="320px"))

def render_new_feat(change=None):
    global _initialized
    key = feat_dd_w2.value
    spec = _FEATURES_W2[key]
    raw_rs = [abs(_airbnb_w2[c].corr(_y_price_w2)) for c in spec["raw_cols"]]
    eng_r = abs(_airbnb_w2[key].corr(_y_price_w2))
    raw_r_text = "  ".join(f"{c} |r|={r:.3f}" for c, r in zip(spec["raw_cols"], raw_rs))

    idx_sample = np.random.RandomState(42).choice(len(_airbnb_w2), 800, replace=False)

    fig = go.Figure(data=go.Scatter(
        x=_airbnb_w2[key].values[idx_sample], y=_y_price_w2.values[idx_sample], mode="markers",
        marker=dict(color=CATEGORY_COLOR[spec["category"]], size=4, opacity=0.4),
    ), layout=base_layout(
        title=f"{key} vs log(price)  |  {raw_r_text}  |  engineered |r|={eng_r:.3f}",
        xaxis_title=key, yaxis_title="log(price)",
    ))

    body = spec["caption"].format(
        r0=raw_rs[0], r1=(raw_rs[1] if len(raw_rs) > 1 else None), eng=eng_r)
    caption = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;background:#f8f9fa;'
        f'padding:12px 16px;border-radius:6px;margin-top:8px;line-height:1.6;color:#333;">'
        f'<p style="margin:0;"><b>{CATEGORY_LABEL[spec["category"]]}:</b> {body}</p></div>'
    )

    with out_new_feat:
        clear_output(wait=True)
        fig.show()
        display(HTML(caption))
    _initialized = True

feat_dd_w2.observe(render_new_feat, names="value")
display(VBox([feat_dd_w2, out_new_feat]))
render_new_feat()

---
## What's happening?

Three patterns for creating new features, and one warning about a fourth kind that only looks new.

**Ratios** normalize one feature by another. `price / (minimum_nights + 1)` gives a per-night cost — more meaningful than raw price for a listing with a 30-night minimum stay. But watch the trap: `availability_ratio = availability_365 / 365` only divides by a *constant*, not another column, and its correlation with price is identical to `availability_365` on its own. Dividing every value by the same number cannot change how those values relate to price.

**Differences** work the same way. `calc_diff = calculated_host_listings_count - 1` subtracts a constant from every row, and it correlates with price exactly as well as `calculated_host_listings_count` does, down past the 15th decimal place. A ratio or a difference only adds signal when the *other* quantity varies — dividing price by minimum_nights works because minimum_nights is not the same number for every row.

**Domain-inspired combinations** combine two columns that each carry real information into one that carries more. `dist_midtown` combines latitude and longitude, using plain Pythagoras, into distance from Times Square. Latitude alone reaches |r| = 0.189, longitude alone reaches |r| = 0.253, and `dist_midtown` reaches |r| = 0.353. One new column beats both of the columns it was built from, because a listing's location isn't really two separate numbers, it's one thing: how close is this to Manhattan?

**And watch for leakage.** `price_per_review = price / (number_of_reviews + 1)` scores |r| = 0.443, by far the highest number on this page. It is also worthless as a feature, because `price` — the thing you're trying to predict — is sitting right inside its own formula.

| Raw features | New feature | Formula | Category | What it means |
|---------------|-------------|---------|----------|----------------|
| `latitude`, `longitude` | `dist_midtown` | `sqrt((lat − 40.758)² + (lon + 73.9855)²)` | Real win | Distance from Times Square; beats both parent columns |
| `availability_365` | `availability_ratio` | `avail / 365` | No change | Dividing by a constant cannot change correlation |
| `calculated_host_listings_count` | `calc_diff` | `count − 1` | No change | Subtracting a constant cannot change correlation either |
| `price`, `number_of_reviews` | `price_per_review` | `price / (reviews + 1)` | Leak | Contains the target; only usable when NOT predicting price |

---
## Real-world example: Which of these features actually helped?

Seven bars, one chart. Three are the raw columns you started with. Four are the engineered features from above. Color marks the category each one landed in: green for a real win, gray for no change, orange for a leak. Only one bar is green and one is orange — the other five, three raw and two engineered, are all gray, because none of them carry any signal about price.

In [3]:
_raw_cols_w2 = ['number_of_reviews', 'availability_365', 'minimum_nights']
_eng_cols_w2 = ['dist_midtown', 'availability_ratio', 'calc_diff', 'price_per_review']
_all_cols_w2 = _raw_cols_w2 + _eng_cols_w2

_category_w2 = {
    'number_of_reviews': 'none', 'availability_365': 'none', 'minimum_nights': 'none',
    'dist_midtown': 'win', 'availability_ratio': 'none', 'calc_diff': 'none',
    'price_per_review': 'leak',
}
_labels_w2 = {
    'number_of_reviews': 'Raw: number_of_reviews',
    'availability_365': 'Raw: availability_365',
    'minimum_nights': 'Raw: minimum_nights',
    'dist_midtown': 'Eng: dist_midtown',
    'availability_ratio': 'Eng: availability_ratio',
    'calc_diff': 'Eng: calc_diff',
    'price_per_review': 'Eng: price_per_review',
}

corrs  = [abs(_airbnb_w2[c].corr(_y_price_w2)) for c in _all_cols_w2]
colors = [CATEGORY_COLOR[_category_w2[c]] for c in _all_cols_w2]
labels = [_labels_w2[c] for c in _all_cols_w2]

fig = go.Figure([go.Bar(
    x=corrs, y=labels, orientation='h',
    marker_color=colors,
    text=[f'{c:.3f}' for c in corrs], textposition='outside',
)])
layout = base_layout(
    title='Which of these features actually helped? (|r| with log(price))',
    xaxis_title='|Pearson correlation|',
    yaxis_title='',
)
layout.update(xaxis=dict(range=[0, max(corrs) * 1.3]), yaxis=dict(autorange='reversed'))
go.Figure(data=fig.data, layout=layout).show()

> **Discussion question:** `price_per_review` uses `price` itself in its formula. Why is that only safe as a feature when predicting something other than price, and what would go wrong if you tried to use it to predict price?

---
## Key takeaway

> **The best new features reveal relationships that already exist in the real world — your job is to translate domain knowledge into arithmetic.**

---
*Next up: 03_binning_and_discretization — where continuous features get converted into stable categorical groups*